# Laboratorio 4 — Simulación de eventos discretos
## 5.4 Simulación de eventos discretos en SimPy

SimPy es un framework de simulación de eventos discretos basado en *procesos*: cada entidad activa es un
generador de Python que cede (`yield`) eventos al entorno (`Environment`), el cual mantiene la lista de
sucesos ordenada por tiempo y despierta a cada proceso cuando su evento ocurre. Los recursos compartidos
(`Resource`, `Container`, `Store`) modelan servidores, depósitos y colas.

Primero se instala el paquete y se recorre el tutorial *SimPy in 10 minutes*; luego se implementan los
siete ejemplos de la documentación oficial (versiones propias, comentadas en español). Para mantener la
coherencia con el resto del laboratorio, toda la aleatoriedad proviene del GLC de módulo $2^{48}$,
inyectado en una subclase de `random.Random`.

In [1]:
import sys, subprocess
try:
    import simpy
except ImportError:                      # en Google Colab SimPy no viene instalado
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'simpy'], check=True)
    import simpy
import random, math, itertools
print("SimPy versión:", simpy.__version__)

SimPy versión: 4.1.2


In [2]:
A, C, M_GLC = 25214903917, 11, 2**48

class GLCRandom(random.Random):
    """random.Random cuya fuente de uniformes es el generador congruencial mixto del Laboratorio 2.
    Al redefinir random() y getrandbits(), todos los métodos (expovariate, uniform, randint, ...) usan el GLC."""
    def __init__(self, X0):
        super().__init__(0)
        self.X = X0; self.consumidos = 0
    def random(self):
        self.X = (A * self.X + C) % M_GLC
        self.consumidos += 1
        return self.X / M_GLC
    def getrandbits(self, k):
        return int(self.random() * (1 << k))
    def seed(self, *a, **kw):
        pass

rng = GLCRandom(2391)
print("Prueba:", [round(rng.random(), 4) for _ in range(5)], rng.randint(1, 6), round(rng.expovariate(1.0), 4))

Prueba: [0.2142, 0.745, 0.661, 0.7595, 0.7835] 4 1.0809


### Tutorial: SimPy en 10 minutos
**(a) Un proceso básico.** Un carro alterna entre estacionado y conduciendo; `env.timeout(d)` es el evento
"han pasado d unidades de tiempo".

In [3]:
def carro(env):
    while True:
        print(f'  t={env.now:4.0f}: empieza a estacionar y cargar')
        yield env.timeout(5)            # duración de la carga
        print(f'  t={env.now:4.0f}: empieza a conducir')
        yield env.timeout(2)            # duración del viaje

env = simpy.Environment()
env.process(carro(env))
env.run(until=15)

  t=   0: empieza a estacionar y cargar
  t=   5: empieza a conducir
  t=   7: empieza a estacionar y cargar
  t=  12: empieza a conducir
  t=  14: empieza a estacionar y cargar


**(b) Interacción entre procesos.** El carro espera a que termine su proceso de carga (`yield proceso`),
pero un conductor puede interrumpirlo (`Interrupt`).

In [4]:
class Carro:
    def __init__(self, env):
        self.env = env
        self.accion = env.process(self.correr())
    def correr(self):
        while True:
            print(f'  t={self.env.now:4.0f}: empieza a estacionar y cargar')
            try:
                yield self.env.process(self.cargar(5))
            except simpy.Interrupt:
                print(f'  t={self.env.now:4.0f}: ¡interrumpido! espero que la batería tenga suficiente carga...')
            print(f'  t={self.env.now:4.0f}: empieza a conducir')
            yield self.env.timeout(2)
    def cargar(self, duracion):
        yield self.env.timeout(duracion)

def conductor(env, carro):
    yield env.timeout(3)
    carro.accion.interrupt()

env = simpy.Environment()
c = Carro(env)
env.process(conductor(env, c))
env.run(until=15)

  t=   0: empieza a estacionar y cargar
  t=   3: ¡interrumpido! espero que la batería tenga suficiente carga...
  t=   3: empieza a conducir
  t=   5: empieza a estacionar y cargar
  t=  10: empieza a conducir
  t=  12: empieza a estacionar y cargar


**(c) Recursos compartidos.** Una estación de carga con dos puntos (`Resource(capacity=2)`); cada carro
solicita (`request`) un punto, espera en cola si están ocupados y lo libera al terminar (`with`).

In [5]:
def carro_bcs(env, nombre, estacion, t_conduce, t_carga):
    yield env.timeout(t_conduce)
    print(f'  t={env.now:3.0f}: {nombre} llega a la estación')
    with estacion.request() as req:
        yield req
        print(f'  t={env.now:3.0f}: {nombre} empieza a cargar')
        yield env.timeout(t_carga)
        print(f'  t={env.now:3.0f}: {nombre} sale de la estación')

env = simpy.Environment()
estacion = simpy.Resource(env, capacity=2)
for i in range(4):
    env.process(carro_bcs(env, f'Carro {i}', estacion, i * 2, 5))
env.run()

  t=  0: Carro 0 llega a la estación
  t=  0: Carro 0 empieza a cargar
  t=  2: Carro 1 llega a la estación
  t=  2: Carro 1 empieza a cargar
  t=  4: Carro 2 llega a la estación
  t=  5: Carro 0 sale de la estación
  t=  5: Carro 2 empieza a cargar
  t=  6: Carro 3 llega a la estación
  t=  7: Carro 1 sale de la estación
  t=  7: Carro 3 empieza a cargar
  t= 10: Carro 2 sale de la estación
  t= 12: Carro 3 sale de la estación


## Ejemplos de la documentación de SimPy
### 1. Bank Renege
Clientes llegan a un banco con un solo cajero; si esperan más de su paciencia (uniforme entre 1 y 3 min)
abandonan la cola (*renege*). Se usa `yield req | env.timeout(paciencia)` para esperar lo que ocurra primero.

In [6]:
def banco_renege(semilla=42, n_clientes=5, intervalo=10.0, paciencia=(1, 3), t_servicio=12.0):
    rng = GLCRandom(semilla)
    env = simpy.Environment()
    cajero = simpy.Resource(env, capacity=1)
    resultados = {'atendidos': 0, 'abandonos': 0}

    def cliente(env, nombre):
        llegada = env.now
        print(f'  {llegada:7.4f} {nombre}: llega')
        with cajero.request() as req:
            pac = rng.uniform(*paciencia)
            res = yield req | env.timeout(pac)
            espera = env.now - llegada
            if req in res:
                print(f'  {env.now:7.4f} {nombre}: esperó {espera:6.3f} y empieza a ser atendido')
                yield env.timeout(rng.expovariate(1.0 / t_servicio))
                print(f'  {env.now:7.4f} {nombre}: termina')
                resultados['atendidos'] += 1
            else:
                print(f'  {env.now:7.4f} {nombre}: ABANDONA tras esperar {espera:6.3f}')
                resultados['abandonos'] += 1

    def fuente(env):
        for i in range(n_clientes):
            env.process(cliente(env, f'Cliente{i:02d}'))
            yield env.timeout(rng.expovariate(1.0 / intervalo))

    env.process(fuente(env))
    env.run()
    print('  Resumen:', resultados)

print('Bank Renege')
banco_renege()

Bank Renege
   0.0000 Cliente00: llega
   0.0000 Cliente00: esperó  0.000 y empieza a ser atendido
   0.0377 Cliente01: llega
   1.8619 Cliente01: ABANDONA tras esperar  1.824
   7.6022 Cliente02: llega
   9.0753 Cliente02: ABANDONA tras esperar  1.473
  15.5777 Cliente03: llega
  18.5335 Cliente03: ABANDONA tras esperar  2.956
  28.6367 Cliente04: llega
  30.4243 Cliente00: termina
  30.4243 Cliente04: esperó  1.788 y empieza a ser atendido
  30.8266 Cliente04: termina
  Resumen: {'atendidos': 2, 'abandonos': 3}


### 2. Carwash
Lavadero con 2 máquinas; los carros llegan cada 5 minutos (uno adicional cada ~5 min tras los 4 iniciales)
y cada lavado dura 5 minutos. El lavadero es un recurso con `capacity=2`.

In [7]:
def carwash(semilla=42, n_maquinas=2, t_lavado=5, t_inter=7, t_sim=20, iniciales=4):
    rng = GLCRandom(semilla)
    env = simpy.Environment()
    class Lavadero:
        def __init__(self, env):
            self.maquina = simpy.Resource(env, n_maquinas)
        def lavar(self, carro):
            yield env.timeout(t_lavado)
            print(f'  t={env.now:3.0f}: el lavadero quitó {rng.randint(50, 99)}% de la suciedad de {carro}')
    def carro(env, nombre, lav):
        print(f'  t={env.now:3.0f}: {nombre} llega al lavadero')
        with lav.maquina.request() as req:
            yield req
            print(f'  t={env.now:3.0f}: {nombre} entra al lavadero')
            yield env.process(lav.lavar(nombre))
            print(f'  t={env.now:3.0f}: {nombre} sale del lavadero')
    def generador(env, lav):
        for i in range(iniciales):
            env.process(carro(env, f'Carro {i}', lav))
        for i in itertools.count(iniciales):
            yield env.timeout(rng.randint(t_inter - 2, t_inter + 2))
            env.process(carro(env, f'Carro {i}', lav))
    lav = Lavadero(env)
    env.process(generador(env, lav))
    env.run(until=t_sim)

print('Carwash')
carwash()

Carwash
  t=  0: Carro 0 llega al lavadero
  t=  0: Carro 1 llega al lavadero
  t=  0: Carro 2 llega al lavadero
  t=  0: Carro 3 llega al lavadero
  t=  0: Carro 0 entra al lavadero
  t=  0: Carro 1 entra al lavadero
  t=  5: Carro 4 llega al lavadero
  t=  5: el lavadero quitó 76% de la suciedad de Carro 0
  t=  5: el lavadero quitó 85% de la suciedad de Carro 1
  t=  5: Carro 0 sale del lavadero
  t=  5: Carro 1 sale del lavadero
  t=  5: Carro 2 entra al lavadero
  t=  5: Carro 3 entra al lavadero
  t= 10: el lavadero quitó 65% de la suciedad de Carro 2
  t= 10: el lavadero quitó 96% de la suciedad de Carro 3
  t= 10: Carro 2 sale del lavadero
  t= 10: Carro 3 sale del lavadero
  t= 10: Carro 4 entra al lavadero
  t= 14: Carro 5 llega al lavadero
  t= 14: Carro 5 entra al lavadero
  t= 15: el lavadero quitó 52% de la suciedad de Carro 4
  t= 15: Carro 4 sale del lavadero
  t= 19: el lavadero quitó 90% de la suciedad de Carro 5
  t= 19: Carro 5 sale del lavadero


### 3. Machine Shop
Taller con 10 máquinas que producen piezas y fallan aleatoriamente (tiempo entre fallas exponencial de
media 300 min). Un único técnico las repara (30 min) pero, mientras no hay máquinas dañadas, hace "otros
trabajos" de menor prioridad que pueden interrumpirse. Usa `PreemptiveResource` e interrupciones.

In [8]:
def machine_shop(semilla=42, n_maquinas=10, t_pieza=10.0, s_pieza=2.0, mttf=300.0, t_rep=30.0,
                 t_otro=30.0, semanas=4):
    rng = GLCRandom(semilla)
    T = semanas * 7 * 24 * 60
    env = simpy.Environment()

    class Maquina:
        def __init__(self, env, nombre, tecnico):
            self.env, self.nombre, self.piezas, self.rota = env, nombre, 0, False
            self.proceso = env.process(self.trabajar(tecnico))
            env.process(self.fallar())
        def trabajar(self, tecnico):
            while True:
                restante = max(0.1, rng.normalvariate(t_pieza, s_pieza))
                while restante:
                    try:
                        inicio = self.env.now
                        yield self.env.timeout(restante)
                        restante = 0
                    except simpy.Interrupt:
                        self.rota = True
                        restante -= self.env.now - inicio
                        with tecnico.request(priority=1) as req:   # prioridad alta: interrumpe otros trabajos
                            yield req
                            yield self.env.timeout(t_rep)
                        self.rota = False
                self.piezas += 1
        def fallar(self):
            while True:
                yield self.env.timeout(rng.expovariate(1.0 / mttf))
                if not self.rota:
                    self.proceso.interrupt()

    def otros_trabajos(env, tecnico):
        while True:
            restante = t_otro
            while restante:
                with tecnico.request(priority=2) as req:            # prioridad baja: puede ser desplazado
                    yield req
                    try:
                        inicio = env.now
                        yield env.timeout(restante)
                        restante = 0
                    except simpy.Interrupt:
                        restante -= env.now - inicio

    tecnico = simpy.PreemptiveResource(env, capacity=1)
    maquinas = [Maquina(env, f'Máquina {i}', tecnico) for i in range(n_maquinas)]
    env.process(otros_trabajos(env, tecnico))
    env.run(until=T)
    print(f'  Resultados del taller tras {semanas} semanas:')
    for m in maquinas:
        print(f'    {m.nombre} fabricó {m.piezas} piezas.')
    print(f'    Total: {sum(m.piezas for m in maquinas)} piezas')

print('Machine Shop')
machine_shop()

Machine Shop


  Resultados del taller tras 4 semanas:
    Máquina 0 fabricó 3309 piezas.
    Máquina 1 fabricó 3247 piezas.
    Máquina 2 fabricó 3323 piezas.
    Máquina 3 fabricó 3348 piezas.
    Máquina 4 fabricó 3339 piezas.
    Máquina 5 fabricó 3426 piezas.
    Máquina 6 fabricó 3337 piezas.
    Máquina 7 fabricó 3324 piezas.
    Máquina 8 fabricó 3398 piezas.
    Máquina 9 fabricó 3265 piezas.
    Total: 33316 piezas


### 4. Movie Renege
Un cine vende boletas de tres películas (`Container` con 50 boletas cada una). Si quedan menos boletas
que las que un grupo quiere comprar, el grupo se va; cuando una película se agota, los que estaban en la
cola por ella reciben el evento `agotada` y abandonan (*renege* colectivo).

In [9]:
def movie_renege(semilla=42, t_sim=120):
    rng = GLCRandom(semilla)
    env = simpy.Environment()
    peliculas = ['Simulación: la película', 'Montecarlo II', 'La cola infinita']
    boletas = {p: simpy.Container(env, 50, init=50) for p in peliculas}
    agotada = {p: env.event() for p in peliculas}
    hora_agotada = {}
    abandonos = {p: 0 for p in peliculas}
    taquilla = simpy.Resource(env, capacity=1)

    def espectador(env, pelicula, n):
        with taquilla.request() as req:
            res = yield req | agotada[pelicula]
            if req not in res:                       # se agotó mientras esperaba
                abandonos[pelicula] += 1
                return
            if boletas[pelicula].level < n:          # no alcanzan para el grupo
                yield env.timeout(0.5)
                return
            yield boletas[pelicula].get(n)
            if boletas[pelicula].level < 2:          # queda(n) una o cero: se declara agotada
                agotada[pelicula].succeed()
                hora_agotada[pelicula] = env.now
                if boletas[pelicula].level > 0:
                    yield boletas[pelicula].get(boletas[pelicula].level)
            yield env.timeout(1)

    def llegadas(env):
        while True:
            yield env.timeout(rng.expovariate(1 / 0.5))
            pel = rng.choice(peliculas)
            n = rng.randint(1, 6)
            if not agotada[pel].triggered:
                env.process(espectador(env, pel, n))

    env.process(llegadas(env))
    env.run(until=t_sim)
    for p in peliculas:
        if agotada[p].triggered:
            print(f'  "{p}" se agotó a los {hora_agotada[p]:.1f} minutos; {abandonos[p]} personas abandonaron la cola.')
        else:
            print(f'  "{p}" no se agotó: quedan {boletas[p].level} boletas.')

print('Movie Renege')
movie_renege()

Movie Renege

  "Simulación: la película" se agotó a los 39.5 minutos; 8 personas abandonaron la cola.
  "Montecarlo II" se agotó a los 36.0 minutos; 6 personas abandonaron la cola.
  "La cola infinita" se agotó a los 29.0 minutos; 13 personas abandonaron la cola.


### 5. Gas Station Refueling
Una gasolinera con 2 surtidores y un tanque (`Container`) de 200 L. Los carros llegan y cargan hasta
llenar su propio tanque; un proceso de control revisa el nivel cada 10 s y, si baja del 10 %, llama al
camión cisterna, que tarda 300 s en llegar y rellena el depósito.

In [10]:
def gas_station(semilla=42, t_sim=2000, cap_estacion=200, umbral=10, tanque_carro=50, nivel=(5, 25),
                tasa=2, t_camion=300, t_inter=(30, 300)):
    rng = GLCRandom(semilla)
    env = simpy.Environment()
    estacion = simpy.Resource(env, 2)
    deposito = simpy.Container(env, cap_estacion, init=cap_estacion)

    def carro(nombre, env):
        nivel_ini = rng.randint(*nivel)
        print(f'  t={env.now:6.1f}: {nombre} llega con {nivel_ini} L')
        with estacion.request() as req:
            inicio = env.now
            yield req
            litros = tanque_carro - nivel_ini
            yield deposito.get(litros)
            yield env.timeout(litros / tasa)
            print(f'  t={env.now:6.1f}: {nombre} terminó de cargar en {env.now - inicio:.1f} s')

    def control(env):
        while True:
            # mejora sobre el ejemplo original: también se llama al camión si hay carros esperando
            # combustible aunque el nivel no haya bajado del umbral (evita el bloqueo del surtidor)
            if deposito.level / cap_estacion * 100 < umbral or deposito.get_queue:
                print(f'  t={env.now:6.1f}: llamando al camión cisterna (nivel {deposito.level:.0f} L, '
                      f'{len(deposito.get_queue)} carro(s) esperando)')
                yield env.process(camion(env))
            yield env.timeout(10)

    def camion(env):
        yield env.timeout(t_camion)
        faltante = deposito.capacity - deposito.level
        print(f'  t={env.now:6.1f}: llegó el camión; rellena {faltante:.0f} L')
        yield deposito.put(faltante)

    def generador(env):
        for i in itertools.count():
            yield env.timeout(rng.randint(*t_inter))
            env.process(carro(f'Carro {i}', env))

    env.process(control(env))
    env.process(generador(env))
    env.run(until=t_sim)

print('Gas Station Refueling')
gas_station()

Gas Station Refueling
  t=  31.0: Carro 0 llega con 22 L
  t=  45.0: Carro 0 terminó de cargar en 14.0 s
  t= 272.0: Carro 1 llega con 18 L
  t= 288.0: Carro 1 terminó de cargar en 16.0 s
  t= 423.0: Carro 2 llega con 25 L
  t= 435.5: Carro 2 terminó de cargar en 12.5 s
  t= 469.0: Carro 3 llega con 6 L
  t= 491.0: Carro 3 terminó de cargar en 22.0 s
  t= 726.0: Carro 4 llega con 12 L
  t= 745.0: Carro 4 terminó de cargar en 19.0 s
  t= 974.0: Carro 5 llega con 13 L
  t= 980.0: llamando al camión cisterna (nivel 33 L, 1 carro(s) esperando)
  t=1151.0: Carro 6 llega con 23 L
  t=1280.0: llegó el camión; rellena 167 L
  t=1293.5: Carro 6 terminó de cargar en 142.5 s
  t=1298.5: Carro 5 terminó de cargar en 324.5 s
  t=1338.0: Carro 7 llega con 10 L
  t=1358.0: Carro 7 terminó de cargar en 20.0 s
  t=1532.0: Carro 8 llega con 13 L
  t=1550.5: Carro 8 terminó de cargar en 18.5 s
  t=1628.0: Carro 9 llega con 17 L
  t=1644.5: Carro 9 terminó de cargar en 16.5 s
  t=1796.0: Carro 10 llega co

### 6. Process Communication
Un generador de mensajes publica en un canal con *broadcast* (cada consumidor recibe una copia a través
de su propio `Store`) y otro canal con un único `Store` compartido en el que cada mensaje lo recibe un solo
consumidor, el primero disponible.

In [11]:
def process_communication(semilla=42, t_sim=100):
    rng = GLCRandom(semilla)
    env = simpy.Environment()

    class CanalBroadcast:
        def __init__(self, env):
            self.env, self.salidas = env, []
        def put(self, valor):
            if not self.salidas:
                raise RuntimeError('sin consumidores')
            return self.env.all_of([s.put(valor) for s in self.salidas])
        def get_output_conn(self):
            s = simpy.Store(self.env)
            self.salidas.append(s)
            return s

    def generador(env, canal):
        for i in itertools.count():
            yield env.timeout(rng.randint(6, 10))
            msg = (env.now, f'Mensaje {i} enviado en t={env.now}')
            canal.put(msg)

    def consumidor(nombre, env, entrada):
        while True:
            msg = yield entrada.get()
            print(f'  t={env.now:3.0f}: {nombre} recibe "{msg[1]}"')

    print('  --- Un mensaje para TODOS los consumidores (broadcast) ---')
    bc = CanalBroadcast(env)
    env.process(generador(env, bc))
    env.process(consumidor('Consumidor A', env, bc.get_output_conn()))
    env.process(consumidor('Consumidor B', env, bc.get_output_conn()))
    env.run(until=t_sim / 2)

    print('  --- Un mensaje para UN solo consumidor (Store compartido) ---')
    env2 = simpy.Environment()
    rng2 = GLCRandom(semilla + 1)
    canal = simpy.Store(env2)
    def generador2(env):
        for i in itertools.count():
            yield env.timeout(rng2.randint(6, 10))
            canal.put((env.now, f'Mensaje {i} enviado en t={env.now}'))
    env2.process(generador2(env2))
    env2.process(consumidor('Consumidor A', env2, canal))
    env2.process(consumidor('Consumidor B', env2, canal))
    env2.run(until=t_sim / 2)

print('Process Communication')
process_communication()

Process Communication
  --- Un mensaje para TODOS los consumidores (broadcast) ---
  t=  6: Consumidor A recibe "Mensaje 0 enviado en t=6"
  t=  6: Consumidor B recibe "Mensaje 0 enviado en t=6"
  t= 16: Consumidor A recibe "Mensaje 1 enviado en t=16"
  t= 16: Consumidor B recibe "Mensaje 1 enviado en t=16"
  t= 25: Consumidor A recibe "Mensaje 2 enviado en t=25"
  t= 25: Consumidor B recibe "Mensaje 2 enviado en t=25"
  t= 35: Consumidor A recibe "Mensaje 3 enviado en t=35"
  t= 35: Consumidor B recibe "Mensaje 3 enviado en t=35"
  t= 42: Consumidor A recibe "Mensaje 4 enviado en t=42"
  t= 42: Consumidor B recibe "Mensaje 4 enviado en t=42"
  --- Un mensaje para UN solo consumidor (Store compartido) ---
  t=  6: Consumidor A recibe "Mensaje 0 enviado en t=6"
  t= 15: Consumidor B recibe "Mensaje 1 enviado en t=15"
  t= 24: Consumidor A recibe "Mensaje 2 enviado en t=24"
  t= 31: Consumidor B recibe "Mensaje 3 enviado en t=31"
  t= 41: Consumidor A recibe "Mensaje 4 enviado en t=41"


### 7. Event Latency
Un cable con retardo: cada mensaje depositado tarda `retardo` unidades de tiempo en estar disponible en
el otro extremo. Se implementa con un `Store` y un proceso auxiliar que espera el retardo antes de
entregar.

In [12]:
def event_latency(semilla=42, retardo=10, t_sim=100):
    rng = GLCRandom(semilla)
    env = simpy.Environment()

    class Cable:
        def __init__(self, env, retardo):
            self.env, self.retardo, self.store = env, retardo, simpy.Store(env)
        def latencia(self, valor):
            yield self.env.timeout(self.retardo)
            self.store.put(valor)
        def put(self, valor):
            self.env.process(self.latencia(valor))
        def get(self):
            return self.store.get()

    def emisor(env, cable):
        while True:
            yield env.timeout(5)
            cable.put(f'Emisor envió esto en t={env.now}')

    def receptor(env, cable):
        while True:
            msg = yield cable.get()
            print(f'  t={env.now:3.0f}: receptor recibió "{msg}"')

    cable = Cable(env, retardo)
    env.process(emisor(env, cable))
    env.process(receptor(env, cable))
    env.run(until=t_sim)

print('Event Latency')
event_latency()

Event Latency
  t= 15: receptor recibió "Emisor envió esto en t=5"
  t= 20: receptor recibió "Emisor envió esto en t=10"
  t= 25: receptor recibió "Emisor envió esto en t=15"
  t= 30: receptor recibió "Emisor envió esto en t=20"
  t= 35: receptor recibió "Emisor envió esto en t=25"
  t= 40: receptor recibió "Emisor envió esto en t=30"
  t= 45: receptor recibió "Emisor envió esto en t=35"
  t= 50: receptor recibió "Emisor envió esto en t=40"
  t= 55: receptor recibió "Emisor envió esto en t=45"
  t= 60: receptor recibió "Emisor envió esto en t=50"
  t= 65: receptor recibió "Emisor envió esto en t=55"
  t= 70: receptor recibió "Emisor envió esto en t=60"
  t= 75: receptor recibió "Emisor envió esto en t=65"
  t= 80: receptor recibió "Emisor envió esto en t=70"
  t= 85: receptor recibió "Emisor envió esto en t=75"
  t= 90: receptor recibió "Emisor envió esto en t=80"
  t= 95: receptor recibió "Emisor envió esto en t=85"
